# NM Assessment Demo

Testing the N-M assessment package components:
1. **nm_assessment.py** - Solves N-M interaction curve
2. **nm_profile.py** - Interactive visualization with eps_bot control

Pattern follows mkappa analysis but for combined N-M loading.

In [ ]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt

from bmcs_cross_section.cs_design.shapes import RectangularShape
from bmcs_cross_section.cs_design.reinforcement import ReinforcementLayout, ReinforcementLayer
from bmcs_cross_section.cs_design.cross_section import CrossSection
from bmcs_cross_section.matmod.ec2_concrete import EC2Concrete
from bmcs_cross_section.matmod.steel_reinforcement import SteelReinforcement
from bmcs_cross_section.nm_assess.nm_assessment import NMAssessment
from bmcs_cross_section.nm_assess.nm_profile import NMProfile
from bmcs_cross_section.cs_design.cs_stress_strain_profile import StressStrainProfile

## 1. Create Cross-Section

Standard rectangular section with reinforcement.

In [ ]:
# Geometry: 300x500 mm rectangular section
shape = RectangularShape(b=300.0, h=500.0)

# Material: C30/37 concrete
concrete = EC2Concrete(f_ck=30.0)

# Reinforcement: Steel B500
steel = SteelReinforcement(f_sy=500.0)

# Bottom layer: 4Ø20 at z=50mm
layer_bottom = ReinforcementLayer(
    z=50.0,
    A_s=1256.6,  # 4×π×10² = 1256.6 mm²
    material=steel
)

# Top layer: 2Ø16 at z=450mm
layer_top = ReinforcementLayer(
    z=450.0,
    A_s=402.1,   # 2×π×8² = 402.1 mm²
    material=steel
)

reinforcement = ReinforcementLayout(layers=[layer_bottom, layer_top])

# Assemble cross-section
cs = CrossSection(
    shape=shape,
    concrete=concrete,
    reinforcement=reinforcement
)

print(f"Cross-section: {cs.shape.b:.0f}×{cs.h_total:.0f} mm")
print(f"Concrete: C{cs.concrete.f_ck:.0f}/37")
print(f"Steel: B{steel.f_sy:.0f}")
print(f"Reinforcement layers: {len(cs.reinforcement.layers)}")

## 2. Test NMAssessment Solver

Solve the N-M interaction curve.

In [ ]:
# Create NM assessment
nm = NMAssessment(cs=cs, n_points=50)

# Solve N-M interaction curve
print("Solving N-M interaction curve...")
nm.solve()
print(f"✓ Solved with {len(nm.N_values)} points")

# Display result ranges
print(f"\nN range: [{nm.N_values.min():.1f}, {nm.N_values.max():.1f}] kN")
print(f"M range: [{nm.M_values.min():.1f}, {nm.M_values.max():.1f}] kNm")
print(f"eps_top range: [{nm.eps_top_values.min()*1000:.2f}, {nm.eps_top_values.max()*1000:.2f}] ‰")
print(f"eps_bot range: [{nm.eps_bot_values.min()*1000:.2f}, {nm.eps_bot_values.max()*1000:.2f}] ‰")

## 3. Plot N-M Interaction Diagram

Visualize the complete interaction curve.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
nm.plot_nm(ax=ax)

# Add design load example
N_Ed = 0.0  # Pure bending
M_Ed = 200.0  # 200 kNm
ax.plot(M_Ed, N_Ed, 'gs', markersize=12, markeredgewidth=2,
        markerfacecolor='green', markeredgecolor='darkgreen',
        label=f'Design Load (N={N_Ed}, M={M_Ed})')

ax.legend()
plt.tight_layout()
plt.show()

## 4. Test StressStrainProfile Directly

Test with specific eps_top and eps_bot values to verify strain visualization.

In [ ]:
# Test case 1: Standard state
eps_top_test = -0.0035  # Concrete ultimate strain (compression)
eps_bot_test = 0.0025   # Steel yielding strain (tension)

# Calculate curvature
h = cs.h_total
kappa_test = (eps_top_test - eps_bot_test) / h

print(f"Test state:")
print(f"  eps_top = {eps_top_test*1000:.2f}‰")
print(f"  eps_bot = {eps_bot_test*1000:.2f}‰")
print(f"  kappa = {kappa_test*1000:.4f} 1/m")

# Create profile
profile = StressStrainProfile(cs, kappa=kappa_test, eps_bottom=eps_bot_test)

# Get forces
F_c, F_s, N_total, M_total = profile.get_force_resultants()
print(f"\nForces:")
print(f"  F_concrete = {F_c/1000:.2f} kN")
print(f"  F_steel = {F_s/1000:.2f} kN")
print(f"  N_total = {N_total/1000:.2f} kN")
print(f"  M_total = {M_total/1e6:.2f} kNm")

In [ ]:
# Verify strain values at top and bottom
eps_at_top = profile.get_strain_at_z(h)
eps_at_bot = profile.get_strain_at_z(0.0)

print(f"Verification:")
print(f"  Strain at top (z={h:.0f}mm): {eps_at_top*1000:.4f}‰ (expected: {eps_top_test*1000:.2f}‰)")
print(f"  Strain at bottom (z=0mm): {eps_at_bot*1000:.4f}‰ (expected: {eps_bot_test*1000:.2f}‰)")
print(f"  Match: {np.isclose(eps_at_top, eps_top_test) and np.isclose(eps_at_bot, eps_bot_test)}")

## 5. Plot Stress-Strain Profile

Visualize the strain and stress distributions.

In [ ]:
# Create figure
fig, (ax_strain, ax_stress) = plt.subplots(
    1, 2, figsize=(14, 6),
    gridspec_kw={'width_ratios': [1, 2], 'wspace': 0},
    sharey=True
)

# Plot profile
profile.plot_stress_strain_profile(
    ax_strain=ax_strain,
    ax_stress=ax_stress,
    show_resultants=True
)

# Add title with strain info
fig.suptitle(
    f'Stress-Strain Profile | ε_top={eps_top_test*1000:.2f}‰, ε_bot={eps_bot_test*1000:.2f}‰, κ={kappa_test*1000:.4f} 1/m',
    fontsize=14, fontweight='bold', y=0.98
)

plt.tight_layout()
plt.show()

print("\n📌 Check: Does the strain profile show correct values?")
print(f"   - Top should show ~{eps_top_test*1000:.2f}‰")
print(f"   - Bottom should show ~{eps_bot_test*1000:.2f}‰")

## 6. Test NMProfile Visualization

Test the interactive profile with different eps_bot values.

In [ ]:
# Create NM profile visualization with specific eps_bot
eps_top = -0.0035  # Fixed top strain
eps_bot = 0.0025   # Adjustable bottom strain

print(f"Creating NM profile with:")
print(f"  eps_top = {eps_top*1000:.2f}‰ (fixed)")
print(f"  eps_bot = {eps_bot*1000:.2f}‰ (adjustable)")

nm_profile = NMProfile(nm, eps_top=eps_top, eps_bot=eps_bot)

# Get current forces
F_c, F_s, N_current, M_current = nm_profile.get_current_forces()
print(f"\nCurrent state forces:")
print(f"  N = {N_current/1000:.2f} kN")
print(f"  M = {M_current/1e6:.2f} kNm")

In [ ]:
# Plot combined visualization
N_Ed = 0.0
M_Ed = 200.0

fig, ax_strain, ax_stress, ax_nm = nm_profile.plot_nm_state(
    figsize=(18, 6),
    show_resultants=True,
    N_Ed=N_Ed,
    M_Ed=M_Ed
)

plt.show()

print("\n📌 Verification:")
print(f"   - Check if strain profile shows ε_top = {eps_top*1000:.2f}‰")
print(f"   - Check if strain profile shows ε_bot = {eps_bot*1000:.2f}‰")
print(f"   - Check if current state marker is on N-M diagram")
print(f"   - Check if design load (green square) is shown at N={N_Ed}, M={M_Ed}")

## 7. Test Different eps_bot Values

Explore different bottom strain values to see the effect.

In [ ]:
# Test case 2: Lower bottom strain (less tension)
eps_bot_test2 = 0.0010

print(f"Test case 2: eps_bot = {eps_bot_test2*1000:.2f}‰")

nm_profile.set_eps_bot(eps_bot_test2)
F_c, F_s, N_current, M_current = nm_profile.get_current_forces()

print(f"  N = {N_current/1000:.2f} kN")
print(f"  M = {M_current/1e6:.2f} kNm")

fig, ax_strain, ax_stress, ax_nm = nm_profile.plot_nm_state(
    figsize=(18, 6),
    show_resultants=True,
    N_Ed=N_Ed,
    M_Ed=M_Ed
)

plt.show()

In [ ]:
# Test case 3: Higher bottom strain (more tension)
eps_bot_test3 = 0.0050

print(f"Test case 3: eps_bot = {eps_bot_test3*1000:.2f}‰")

nm_profile.set_eps_bot(eps_bot_test3)
F_c, F_s, N_current, M_current = nm_profile.get_current_forces()

print(f"  N = {N_current/1000:.2f} kN")
print(f"  M = {M_current/1e6:.2f} kNm")

fig, ax_strain, ax_stress, ax_nm = nm_profile.plot_nm_state(
    figsize=(18, 6),
    show_resultants=True,
    N_Ed=N_Ed,
    M_Ed=M_Ed
)

plt.show()

## 8. Test with Compression at Bottom

Test case where bottom fiber is in compression (negative strain).

In [ ]:
# Test case 4: Compression at bottom (axial load case)
eps_bot_test4 = -0.0010  # Compression

print(f"Test case 4: eps_bot = {eps_bot_test4*1000:.2f}‰ (compression)")

nm_profile.set_eps_bot(eps_bot_test4)
F_c, F_s, N_current, M_current = nm_profile.get_current_forces()

print(f"  N = {N_current/1000:.2f} kN (should be negative = compression)")
print(f"  M = {M_current/1e6:.2f} kNm")

fig, ax_strain, ax_stress, ax_nm = nm_profile.plot_nm_state(
    figsize=(18, 6),
    show_resultants=True,
    N_Ed=-500.0,  # Design compression load
    M_Ed=100.0
)

plt.show()

print("\n📌 Check: Entire section should be in compression (blue strain profile)")

## Summary

**Key Points to Verify:**

1. **Strain Profile Accuracy**
   - Does the strain profile show the correct eps_top value at the top?
   - Does the strain profile show the correct eps_bot value at the bottom?
   - Does the curvature indicator (κ) match the calculated value?

2. **NMProfile Behavior**
   - Does changing eps_bot update the visualization correctly?
   - Does the current state marker move on the N-M diagram?
   - Are the forces (N, M) computed correctly?

3. **Consistency**
   - Do the strain values in the title match the slider/input values?
   - Does the profile visualization match the numerical output?

If any discrepancies are found, they can be traced to:
- StressStrainProfile initialization
- NMProfile._update_profile() method
- Plot title generation